# 节点 04：LeNet — 从零手撕卷积神经网络

本 notebook 只用 **numpy**，从零实现：
1. 2D 卷积
2. Max pooling
3. 一个迷你 CNN（卷积 + 池化 + 全连接）
4. 参数量对比

> 前置：你已读过节点 03（反向传播），知道梯度和链式法则是什么。

In [1]:
import numpy as np
np.random.seed(42)

## Part 1：手写 2D 卷积

### 用一个 3x3 图片和一个 2x2 kernel 演示

In [2]:
def conv2d(image, kernel):
    """2D 卷积，不含 padding，步长=1
    
    image:  (H, W) 矩阵
    kernel: (kH, kW) 矩阵
    输出:   (H-kH+1, W-kW+1) 矩阵
    """
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            # 取图片的局部区域
            patch = image[i:i+kH, j:j+kW]
            # 对应位置相乘再求和
            output[i, j] = np.sum(patch * kernel)
    return output


# 测试：用一个简单的竖直边缘 kernel
image = np.array([
    [100, 100,  0,  0],
    [100, 100,  0,  0],
    [100, 100,  0,  0],
    [100, 100,  0,  0],
], dtype=float)

# 竖直边缘探测 kernel：左亮右暗 → 输出大；均匀区域 → 输出接近 0
edge_kernel = np.array([
    [ 1,  0, -1],
    [ 2,  0, -2],
    [ 1,  0, -1],
], dtype=float)

result = conv2d(image, edge_kernel)
print("输入图片（左半亮、右半暗）:")
print(image)
print()
print("卷积结果（边缘在第 2 列，值最大）:")
print(result)

输入图片（左半亮、右半暗）:
[[100. 100.   0.   0.]
 [100. 100.   0.   0.]
 [100. 100.   0.   0.]
 [100. 100.   0.   0.]]

卷积结果（边缘在第 2 列，值最大）:
[[400. 400.]
 [400. 400.]]


In [3]:
# 测试：用一个 6×4 图片（左半均匀、右半均匀、中间是边缘）
# 这样卷积能清楚展示：均匀区域输出 ≈ 0，边缘区域输出大

image6 = np.array([
    [100, 100, 100,  0,  0,  0],
    [100, 100, 100,  0,  0,  0],
    [100, 100, 100,  0,  0,  0],
    [100, 100, 100,  0,  0,  0],
    [100, 100, 100,  0,  0,  0],
], dtype=float)

result6 = conv2d(image6, edge_kernel)
print("输出形状:", result6.shape)  # 应为 (3, 4)
print("卷积结果:")
print(result6.astype(int))
print()
print("列 0 (均匀区域，输出应≈0):", result6[0, 0])
print("列 2 (边缘处，输出应很大):", result6[0, 2])

assert result6.shape == (3, 4), f"卷积输出形状错误: {result6.shape}"
assert result6[0, 2] > result6[0, 0], "边缘处应比均匀区域大"
assert result6[0, 0] == 0, "均匀亮区内部的卷积响应应为 0"
print("卷积函数测试通过!")


输出形状: (3, 4)
卷积结果:
[[  0 400 400   0]
 [  0 400 400   0]
 [  0 400 400   0]]

列 0 (均匀区域，输出应≈0): 0.0
列 2 (边缘处，输出应很大): 400.0
卷积函数测试通过!


## Part 2：手写 Max Pooling

In [4]:
def max_pool2d(feature_map, pool_size=2):
    """Max pooling，步长 = pool_size（不重叠）
    
    feature_map: (H, W)
    输出:        (H//pool_size, W//pool_size)
    """
    H, W = feature_map.shape
    pH = H // pool_size
    pW = W // pool_size
    
    output = np.zeros((pH, pW))
    for i in range(pH):
        for j in range(pW):
            region = feature_map[i*pool_size:(i+1)*pool_size,
                                  j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(region)
    return output


# 测试
test_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [9, 2, 1, 3],
    [0, 4, 5, 6],
], dtype=float)

pooled = max_pool2d(test_map, pool_size=2)
print("原始 feature map (4x4):")
print(test_map)
print()
print("Max Pooling 后 (2x2):")
print(pooled)

expected = np.array([[6, 8], [9, 6]])
assert np.array_equal(pooled, expected), f"Max pooling 结果错误: {pooled}"
print("Max Pooling 测试通过!")

原始 feature map (4x4):
[[1. 3. 2. 4.]
 [5. 6. 7. 8.]
 [9. 2. 1. 3.]
 [0. 4. 5. 6.]]

Max Pooling 后 (2x2):
[[6. 8.]
 [9. 6.]]
Max Pooling 测试通过!


## Part 3：参数量对比

全连接 vs 卷积+池化，用 28x28 图片识别 10 类数字为例

In [5]:
def count_params_fc(input_size, hidden_size, output_size):
    """全连接网络参数量：输入→隐藏→输出"""
    return input_size * hidden_size + hidden_size * output_size

def count_params_cnn(img_h, img_w, kernels1, kernel_size1,
                     kernels2, kernel_size2, pool_size,
                     fc_hidden, output_size):
    """简单 CNN 参数量计算"""
    # 卷积层 1
    # 每个 kernel 有 kernel_size1^2 个权重，共 kernels1 个
    c1_params = kernel_size1 * kernel_size1 * kernels1
    # 卷积后尺寸
    h1 = img_h - kernel_size1 + 1  # 24
    w1 = img_w - kernel_size1 + 1  # 24
    # 池化后尺寸
    h1p = h1 // pool_size  # 12
    w1p = w1 // pool_size  # 12
    
    # 卷积层 2（输入 channels = kernels1）
    c2_params = kernel_size2 * kernel_size2 * kernels1 * kernels2
    h2 = h1p - kernel_size2 + 1  # 8
    w2 = w1p - kernel_size2 + 1  # 8
    h2p = h2 // pool_size  # 4
    w2p = w2 // pool_size  # 4
    
    # 展平后的大小
    flat_size = h2p * w2p * kernels2  # 4*4*16 = 256
    
    # 全连接层
    fc_params = flat_size * fc_hidden + fc_hidden * output_size
    
    total = c1_params + c2_params + fc_params
    return {
        'conv1': c1_params,
        'conv2': c2_params,
        'fc': fc_params,
        'total': total,
        'flat_size': flat_size,
    }


# 全连接网络：784 → 1000 → 10
fc_params = count_params_fc(784, 1000, 10)
print(f"全连接网络 (784→1000→10): {fc_params:,} 个参数")

# LeNet-5 风格：28x28 → conv(6,5x5) → pool(2) → conv(16,5x5) → pool(2) → 120 → 10
cnn_info = count_params_cnn(28, 28, 6, 5, 16, 5, 2, 120, 10)
print(f"\nLeNet-5 风格 CNN:")
print(f"  卷积层 1 (6个5x5 kernel): {cnn_info['conv1']:,} 个参数")
print(f"  卷积层 2 (16个5x5 kernel): {cnn_info['conv2']:,} 个参数")
print(f"  全连接层 ({cnn_info['flat_size']}→120→10): {cnn_info['fc']:,} 个参数")
print(f"  总计: {cnn_info['total']:,} 个参数")
print(f"\n参数量比值: 全连接/CNN = {fc_params / cnn_info['total']:.1f}x")

assert fc_params > cnn_info['total'], "CNN 参数量应远小于全连接"
print("\n参数量对比测试通过!")

全连接网络 (784→1000→10): 794,000 个参数

LeNet-5 风格 CNN:
  卷积层 1 (6个5x5 kernel): 150 个参数
  卷积层 2 (16个5x5 kernel): 2,400 个参数
  全连接层 (256→120→10): 31,920 个参数
  总计: 34,470 个参数

参数量比值: 全连接/CNN = 23.0x

参数量对比测试通过!


## Part 4：迷你 CNN 训练

用一个简化版 CNN（只有 1 个 kernel + max pooling + 全连接）
训练识别一个玩具任务：
- 输入：4x4 的图片，左半全 1 或右半全 1
- 输出：0 = 左半亮，1 = 右半亮

**目标**：观察 CNN 训练后准确率能否到 100%，并分析「kernel 学到了什么」。

> **提前剧透**：你会发现 kernel 学出来是全正权重（亮度增强器），不是方向性边缘探测器。
> 这不是 bug——而是一个重要的发现：位置区分的任务被 FC 层接手了，kernel 负责放大亮度信号。
> 这揭示了 CNN 里「分工」的本质：低层 kernel 检测局部信号强度，高层 FC 做空间推理。

In [6]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -100, 100)))


class MiniCNN:
    """极简 CNN：1个 2x2 kernel -> relu -> flatten(9) -> FC(9->1) -> sigmoid

    输入: (4, 4) 图片
    卷积后: (3, 3) 特征图（4-2+1=3）
    展平: 9 个数
    全连接: FC(9->1) -> sigmoid -> 预测

    注意：我们保留了完整的 3x3 特征图（不做全局池化），
    这样 FC 层能「感知位置」——左边亮还是右边亮，feature map 不同位置的值不同。
    """
    def __init__(self):
        self.kernel = np.random.randn(2, 2) * 0.1
        self.W = np.random.randn(9) * 0.1  # FC 权重：9 输入 -> 1 输出
        self.b = 0.0

    def forward(self, image):
        self.image = image
        self.conv = conv2d(image, self.kernel)    # (3, 3)
        self.relu = np.maximum(0, self.conv)       # (3, 3) ReLU
        self.flat = self.relu.flatten()            # (9,)
        self.fc_in = np.dot(self.flat, self.W) + self.b
        self.pred = sigmoid(self.fc_in)
        return self.pred

    def backward(self, image, y, lr=0.05):
        pred = self.pred

        # dL/dpred，L = (1/2)(y-pred)^2 -> 导数 = -(y-pred)
        dL_dpred = -(y - pred)
        # 通过 sigmoid
        dL_dfc = dL_dpred * pred * (1 - pred)
        # 通过全连接
        dL_dW = dL_dfc * self.flat
        dL_db = dL_dfc
        dL_dflat = dL_dfc * self.W
        # 通过 flatten（reshape 回 3x3）
        dL_drelu = dL_dflat.reshape(3, 3)
        # 通过 ReLU
        dL_dconv = dL_drelu * (self.conv > 0).astype(float)
        # 通过卷积：计算 kernel 梯度
        kH, kW = self.kernel.shape
        dL_dkernel = np.zeros_like(self.kernel)
        for i in range(dL_dconv.shape[0]):
            for j in range(dL_dconv.shape[1]):
                dL_dkernel += dL_dconv[i, j] * image[i:i+kH, j:j+kW]
        # 更新参数
        self.W -= lr * dL_dW
        self.b -= lr * dL_db
        self.kernel -= lr * dL_dkernel
        return 0.5 * (y - pred) ** 2


print('MiniCNN 定义完成')


MiniCNN 定义完成


In [7]:
# 生成训练数据
def make_data(n=100):
    """生成 4x4 图片：左半亮=标签0，右半亮=标签1"""
    images = []
    labels = []
    for _ in range(n):
        label = np.random.randint(0, 2)
        img = np.zeros((4, 4))
        if label == 0:
            img[:, :2] = 1.0  # 左半亮
        else:
            img[:, 2:] = 1.0  # 右半亮
        # 加一点噪声
        img += np.random.randn(4, 4) * 0.1
        images.append(img)
        labels.append(float(label))
    return images, labels

X_train, y_train = make_data(200)

# 可视化前 4 个样本
print("训练数据样例:")
for i in range(4):
    print(f"  样本 {i}: 标签={y_train[i]:.0f} ({'右半亮' if y_train[i] else '左半亮'})")
    print(f"  图片:\n{np.round(X_train[i], 1)}")

训练数据样例:
  样本 0: 标签=0 (左半亮)
  图片:
[[ 0.9  1.1  0.   0.1]
 [ 0.9  1.  -0.2 -0. ]
 [ 1.   0.9  0.  -0. ]
 [ 1.   1.  -0.1 -0. ]]
  样本 1: 标签=1 (右半亮)
  图片:
[[-0.1 -0.2  0.9  1.1]
 [ 0.3  0.   1.   0.9]
 [-0.1  0.1  1.   0.9]
 [-0.1 -0.   0.9  1. ]]
  样本 2: 标签=1 (右半亮)
  图片:
[[-0.1 -0.   1.   1.2]
 [ 0.  -0.1  0.8  0.9]
 [ 0.   0.   1.1  0.8]
 [ 0.1  0.   1.   1.1]]
  样本 3: 标签=0 (左半亮)
  图片:
[[ 1.   1.1  0.  -0.2]
 [ 1.   1.  -0.1  0.1]
 [ 1.1  1.1 -0.1 -0. ]
 [ 1.   1.1 -0.  -0. ]]


In [8]:
# 训练 MiniCNN
model = MiniCNN()
losses = []

print('初始 kernel:')
print(model.kernel)
print()

for epoch in range(150):
    epoch_loss = 0
    for img, label in zip(X_train, y_train):
        pred = model.forward(img)
        loss = model.backward(img, label, lr=0.05)
        epoch_loss += loss
    avg_loss = epoch_loss / len(X_train)
    losses.append(avg_loss)
    if epoch % 30 == 0:
        correct = sum(
            1 for img, label in zip(X_train, y_train)
            if round(model.forward(img)) == label
        )
        acc = correct / len(X_train)
        print(f'Epoch {epoch:3d}: loss={avg_loss:.4f}, acc={acc:.2%}')

print()
print('训练后 kernel（全正权重 = 亮度增强器，不是方向探测器）:')
print(np.round(model.kernel, 3))


初始 kernel:
[[0.01847105 0.09379155]
 [0.00124993 0.28684031]]

Epoch   0: loss=0.0710, acc=100.00%


Epoch  30: loss=0.0000, acc=100.00%


Epoch  60: loss=0.0000, acc=100.00%


Epoch  90: loss=0.0000, acc=100.00%


Epoch 120: loss=0.0000, acc=100.00%



训练后 kernel（全正权重 = 亮度增强器，不是方向探测器）:
[[0.493 0.909]
 [0.476 1.09 ]]


In [9]:
# 验证最终准确率
X_test, y_test = make_data(100)
correct = sum(
    1 for img, label in zip(X_test, y_test)
    if round(model.forward(img)) == label
)
final_acc = correct / len(X_test)
print(f'测试集准确率: {final_acc:.2%}')

assert final_acc >= 0.75, f'准确率太低: {final_acc:.2%}（期望 >= 75%）'
print('\nMiniCNN 测试通过! CNN 成功学会区分左半亮和右半亮。')


测试集准确率: 100.00%

MiniCNN 测试通过! CNN 成功学会区分左半亮和右半亮。


### 分析：kernel 学到了什么？

训练后的 kernel 通常长这样（具体数值每次训练不同，但模式相似）：

```
[[+0.5  +0.9]
 [+0.5  +1.1]]
```

**全部是正数**。这意味着：kernel 是个「亮度放大器」，哪里亮就输出大，哪里暗就输出小。

**为什么不是「右边探测器」？**

因为我们给了 FC 层 9 个权重（对应 3×3 特征图的 9 个位置），
FC 层完全可以通过「特征图右侧位置的值更大」来区分左右——
不需要 kernel 有方向性，FC 层自己就能做位置推理。

**这是 CNN 的一个关键洞察**：

| 层 | 职责 |
|---|---|
| **Kernel（卷积层）** | 检测局部特征（这里是亮度强度） |
| **FC 层** | 用特征图的空间位置做最终判断 |

真正的边缘探测（左正右负的 kernel）会在「找边界」任务中自然出现，
而不是「判断哪半亮」的任务。任务决定 kernel 长什么样。

In [10]:
# 损失曲线（文字版，不依赖 matplotlib）
print('训练损失曲线（每 30 轮一个点）:')
sampled = losses[::30]
max_val = max(sampled) if max(sampled) > 0 else 1
for i, v in enumerate(sampled):
    bar_len = int(v / max_val * 30)
    print(f'Epoch {i*30:3d}: {"#" * bar_len:<30} {v:.4f}')
print('\n损失持续下降 = 模型在学习 :)')


训练损失曲线（每 30 轮一个点）:
Epoch   0: ############################## 0.0710
Epoch  30:                                0.0000
Epoch  60:                                0.0000
Epoch  90:                                0.0000
Epoch 120:                                0.0000

损失持续下降 = 模型在学习 :)


## 总结

你刚刚从零实现了一个能「自动学习特征」的卷积神经网络：

1. **卷积** = 一个小探测器扫描整张图片（参数少、懂空间结构）
2. **Max Pooling** = 缩小图片，保留重要信息
3. **反向传播** 同样适用于 CNN，只是需要多算一步「卷积的梯度」

LeNet 的贡献：把这三个思想组合成一个端到端可训练的架构，用于真实的图片识别任务。

下一站：[节点 05 — AlexNet（2012）](../docs/05-alexnet-2012.md)，看这个架构如何在 20 年后引爆深度学习革命。